In [17]:
import soundfile as sf
import io
import numpy as np
import base64
import requests
import subprocess
import librosa
from IPython.display import Audio

In [18]:
SAMPLE_RATE = 16000
URL = "https://api.sangjeong.com:8080/whisper/stt_duration"
URL = "http://localhost:8000/whisper/stt_duration"

In [19]:
def compress_to_opus(bytes: bytes) -> bytes:
  process = subprocess.Popen(
    ["ffmpeg", "-i", "pipe:0", "-c:a", "libopus", "-f", "opus", "pipe:1"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE # subprocess.DEVNULL 하면 속도 조금 더 빨라짐
  )

  out, err = process.communicate(input=bytes)
  return out, err 

In [20]:
def np_to_wav(audio: np.ndarray, sample_rate:int) -> bytes:
  buffer = io.BytesIO()
  sf.write(buffer, audio, sample_rate, format='wav')
  return buffer.getvalue()

In [21]:
audio, sr = librosa.load(".data/news_with_english.mp3", sr=SAMPLE_RATE)

In [22]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=4000, scale=400))
  rand_len = np.clip(rand_len, 3500, 4500)
  end = min(pos + rand_len, total_samples)

  chunk = audio[pos:end]
  segments.append(chunk)
  pos = end

In [23]:
idx = 0

In [24]:
audio = segments[idx]
idx += 1
Audio(audio, rate=SAMPLE_RATE)

In [ ]:
output = []
for segment in segments:
  audio = np_to_wav(segment, SAMPLE_RATE)
  audio, _ = compress_to_opus(audio)
  audio_b64 = base64.b64encode(audio).decode('utf-8')
  params = {
    "group": "1", 
    "user": "1", 
    "audio": audio_b64
  }

  res = requests.get(URL, params=params)
  # print(res)
  d = res.json()
  # print(d)
  output = [ *output, *d['completed'] ]

In [29]:
for s in output:
  print(s["text"])
  print(f"\t start:{s['words'][0]['start']}, end:{s['words'][-1]['end']}")

BBC 생방송 인터뷰 중에 자녀 난입 사건으로 스타가 된 미국인 교수 가족이 오늘 카메라 앞에 섰습니다.
	 start:2880.0, end:124115.0
유튜브 스타가 된 4살짜리 딸은 이번엔 사탕을 입에 물고 등장했습니다.
	 start:130835.0, end:210802.0
배영진 기자입니다.
	 start:217842.0, end:234802.0
딸과 아들의 등장으로 일약 스타가 된 로버트 켈리 부산대 교수.
	 start:301245.0, end:371645.0
유튜브 영상 조회비가,600만 건이 넘는 켈리 교수 가족은 세계적인 유명 인사가 됐습니다.
	 start:377725.0, end:477967.0
이렇게 언론의 관심이 커지자 켈리 교수 가족이 기자회견에 나섰습니다.
	 start:484047.0, end:561415.0
춤을 췄던 첫째 딸 메리아는 사탕을 물었고 둘째 아들 존은 엄마 품에 안긴 모습이었습니다. 
	 start:570535.0, end:751766.0
thought it was a disaster. 
	 start:763336.0, end:779016.0
I immediately called or texted or emailed the BBC. 
	 start:784136.0, end:836977.0
I communicated with the BBC immediately afterwards, and I apologized to them.
	 start:838257.0, end:891864.0
I said that if they never us back or never asked me to be on television again, I would understand.
	 start:892505.0, end:971993.0
귀여운 춤으로 화제가 된 딸에 대한 질문도 잇따랐습니다.
	 start:982016.0, end:1032911.0
당시 BBC와 인터뷰한 내용은 한국의 대통령 탄핵 사건이었습니다. 
	 start:113825